# Neptune AgentCore Gateway - Local Testing

Tests the deployed AgentCore Gateway with its Amazon Neptune tools.

**Tools available (9):**
- `discover_named_graphs` - List all named graphs
- `get_ontology_from_neptune` - Retrieve full ontology by `ontology_id` (UUID)
- `persist_to_neptune` - Write RDF n-quad data
- `delete_graph` - Drop all triples in a named graph by `ontology_id`
- `execute_sparql_query` - Execute generic SPARQL queries
- `get_graph_summary` - Summary statistics
- `get_graph_stats` - Class distribution statistics
- `get_graph_classes` - List all classes
- `get_graph_properties` - List all properties

## Step 1: Install dependencies

In [1]:
!pip install boto3 mcp-proxy-for-aws strands-agents --quiet


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
# Configure boto3 session with AWS profile
import boto3
from botocore.config import Config
from dotenv import load_dotenv
import os

config = Config(
    retries={
        'max_attempts': 10,
        'mode': 'standard'
    },
    signature_version='v4'
)

# Load environment variables from .env file (falls back to os.environ if not found)
load_dotenv(dotenv_path='.env', override=False)

# Create session with specific AWS profile
session = boto3.Session(profile_name=os.environ.get('AWS_PROFILE', 'XXX'))
region = session.region_name or 'us-east-1'

# Create STS client to verify credentials
sts = session.client(
    service_name='sts',
    region_name=region,
    config=config
)

# Verify credentials work
try:
    identity = sts.get_caller_identity()
    print(f"✓ AWS Credentials Verified:")
    print(f"  Profile: {os.environ.get('AWS_PROFILE', 'XXX')}")
    print(f"  Account: {identity['Account'][-4:]}")
    print(f"  User/Role: {identity['Arn'].split('/')[-1]}")
    print(f"  Region: {region}")
except Exception as e:
    print(f"✗ Failed to verify AWS credentials: {str(e)}")
    raise

✓ AWS Credentials Verified:
  Profile: huthmac-demo
  Account: 381492284087
  User/Role: huthmac-Isengard
  Region: us-east-1


## Step 2: Load Gateway configuration from SSM

In [ ]:
import boto3
import os


PROJECT_NAME = os.environ.get('PROJECT_NAME', 'XXX')



ssm = session.client(
    service_name='ssm',
    region_name=region,
    config=config
)

def get_ssm_param(name):
    resp = ssm.get_parameter(Name=name)
    return resp['Parameter']['Value']

GATEWAY_URL = get_ssm_param(f'/{PROJECT_NAME}/neptune-gateway/url')
GATEWAY_ID  = get_ssm_param(f'/{PROJECT_NAME}/neptune-gateway/id')

print(f'Gateway URL : {GATEWAY_URL}')
print(f'Gateway ID  : {GATEWAY_ID}')

Gateway URL : https://semantic-layer-neptune-gateway-us71stsoec.gateway.bedrock-agentcore.us-east-1.amazonaws.com/mcp
Gateway ID  : semantic-layer-neptune-gateway-us71stsoec


## Step 3: List available tools

Verify the Gateway is reachable and all 8 Neptune tools are registered.

In [4]:
from strands.tools.mcp import MCPClient
from mcp_proxy_for_aws.client import aws_iam_streamablehttp_client

# Pass credentials from the already-authenticated huthmac-demo session so that
# aws_iam_streamablehttp_client uses SigV4 IAM signing with those credentials,
# rather than creating a new default boto3 session that picks up expired SSO tokens.
mcp_client = MCPClient(
    lambda: aws_iam_streamablehttp_client(
        endpoint=GATEWAY_URL,
        aws_region=region,
        aws_service='bedrock-agentcore',
        credentials=session.get_credentials(),
    )
)

with mcp_client:
    tools = mcp_client.list_tools_sync()

print(f'Found {len(tools)} tools:')
for t in tools:
    print(f'  - {t.tool_name}')


/Users/huthmac/.pyenv/versions/3.12.8/lib/python3.12/contextlib.py:105: DeprecationWarning: Use `streamable_http_client` instead.
  self.gen = func(*args, **kwds)


Found 9 tools:
  - delete-graph___delete_graph
  - discover-named-graphs___discover_named_graphs
  - execute-sparql-query___execute_sparql_query
  - get-graph-classes___get_graph_classes
  - get-graph-properties___get_graph_properties
  - get-graph-stats___get_graph_stats
  - get-graph-summary___get_graph_summary
  - get-ontology-from-neptune___get_ontology_from_neptune
  - persist-to-neptune___persist_to_neptune


## Step 4: Individual tool invocations

In [5]:
import uuid, json

# Cache: short tool name → full MCP name registered by the Gateway
# e.g. 'discover_named_graphs' → 'discover-named-graphs___discover_named_graphs'
_tool_name_cache: dict = {}

def call_tool(short_name: str, arguments: dict | None = None):
    """
    Invoke a Neptune Gateway tool by its short name (e.g. 'discover_named_graphs').

    On the first call the helper connects, lists all tools, and builds a mapping
    from the suffix after '___' to the full Gateway-prefixed name that the MCP
    protocol requires.  Subsequent calls reuse the cached mapping.
    """
    global _tool_name_cache

    with mcp_client:
        # Populate cache once per kernel session
        if not _tool_name_cache:
            tools = mcp_client.list_tools_sync()
            for t in tools:
                full  = t.mcp_tool.name                       # 'discover-named-graphs___discover_named_graphs'
                short = full.split('___')[-1].replace('-', '_') # 'discover_named_graphs'
                _tool_name_cache[short] = full
            print(f'Tool name cache built: {list(_tool_name_cache.keys())}')

        full_name = _tool_name_cache.get(short_name)
        if full_name is None:
            raise ValueError(f"Unknown tool '{short_name}'. Available: {list(_tool_name_cache.keys())}")

        result = mcp_client.call_tool_sync(
            tool_use_id=str(uuid.uuid4()),
            name=full_name,
            arguments=arguments or {},
        )

    for item in result['content']:
        raw = item.get('text') or item.get('json')
        if raw is None:
            raw = str(item)
        if isinstance(raw, (dict, list)):
            print(json.dumps(raw, indent=2))
        else:
            try:
                print(json.dumps(json.loads(raw), indent=2))
            except (json.JSONDecodeError, TypeError):
                print(str(raw))


In [6]:
# ── discover_named_graphs ────────────────────────────────────────────────────
# Lists all named graphs (ontologies) stored in the Neptune database.
# No input parameters required.

call_tool('discover_named_graphs')


Tool name cache built: ['delete_graph', 'discover_named_graphs', 'execute_sparql_query', 'get_graph_classes', 'get_graph_properties', 'get_graph_stats', 'get_graph_summary', 'get_ontology_from_neptune', 'persist_to_neptune']
{
  "statusCode": 200,
  "body": "{\n  \"named_graphs\": [\n    \"http://example.com/f972955d-db45-4bac-968a-80e0c3d16d4c/graph\",\n    \"http://example.com/f972955d-db45-4bac-968a-80e0c3d16d4c/ontology\",\n    \"http://example.com/f972955d-db45-4bac-968a-80e0c3d16d4c/ontology/1.0.0\",\n    \"http://example.com/ontology/f972955d-db45-4bac-968a-80e0c3d16d4c\",\n    \"http://example.com/test/graph\",\n    \"urn:graph:f972955d-db45-4bac-968a-80e0c3d16d4c\",\n    \"urn:graph:ontology:f972955d-db45-4bac-968a-80e0c3d16d4c\",\n    \"urn:ontology:f972955d-db45-4bac-968a-80e0c3d16d4c\"\n  ],\n  \"count\": 8\n}"
}


In [7]:
# ── execute_sparql_query ─────────────────────────────────────────────────────
# Executes a generic SPARQL query (SELECT or UPDATE) against Neptune.

sparql_query = 'SELECT ?s ?p ?o WHERE { ?s ?p ?o } LIMIT 10'
query_type   = 'SELECT'   # 'SELECT' or 'UPDATE'

call_tool('execute_sparql_query', {
    'sparql_query': sparql_query,
    'query_type':   query_type,
})


{
  "statusCode": 200,
  "body": "{\"head\": {\"vars\": [\"s\", \"p\", \"o\"]}, \"results\": {\"bindings\": [{\"s\": {\"type\": \"uri\", \"value\": \"http://example.com/ontology/f972955d-db45-4bac-968a-80e0c3d16d4c/InvestProduct/sk\"}, \"p\": {\"type\": \"uri\", \"value\": \"http://www.w3.org/1999/02/22-rdf-syntax-ns#type\"}, \"o\": {\"type\": \"uri\", \"value\": \"http://www.w3.org/2002/07/owl#DatatypeProperty\"}}, {\"s\": {\"type\": \"uri\", \"value\": \"http://example.com/ontology/f972955d-db45-4bac-968a-80e0c3d16d4c/InvestProduct/sk\"}, \"p\": {\"type\": \"uri\", \"value\": \"http://www.w3.org/2000/01/rdf-schema#label\"}, \"o\": {\"type\": \"literal\", \"value\": \"sort key\"}}, {\"s\": {\"type\": \"uri\", \"value\": \"http://example.com/ontology/f972955d-db45-4bac-968a-80e0c3d16d4c/InvestProduct/sk\"}, \"p\": {\"type\": \"uri\", \"value\": \"http://www.w3.org/2000/01/rdf-schema#comment\"}, \"o\": {\"type\": \"literal\", \"value\": \"Sort key for the investment product record.\"}},

In [8]:
# ── get_graph_summary ────────────────────────────────────────────────────────
# Returns class count, property count, and triple count for an ontology graph.

ontology_id = 'f972955d-db45-4bac-968a-80e0c3d16d4c'   # replace with a graph name from discover_named_graphs

call_tool('get_graph_summary', {
    'ontology_id': ontology_id,
})


{
  "statusCode": 200,
  "body": "{\"ontologyId\": \"f972955d-db45-4bac-968a-80e0c3d16d4c\", \"graphUri\": \"http://example.com/ontology/f972955d-db45-4bac-968a-80e0c3d16d4c\", \"classCount\": 12, \"propertyCount\": 46, \"tripleCount\": 378}"
}


In [9]:
# ── get_graph_stats ──────────────────────────────────────────────────────────
# Returns the top-20 classes by instance count for an ontology graph.


call_tool('get_graph_stats', {
    'ontology_id': ontology_id,
})


{
  "statusCode": 200,
  "body": "{\"ontologyId\": \"f972955d-db45-4bac-968a-80e0c3d16d4c\", \"graphUri\": \"http://example.com/ontology/f972955d-db45-4bac-968a-80e0c3d16d4c\", \"classDistribution\": []}"
}


In [10]:
# ── get_graph_classes ────────────────────────────────────────────────────────
# Returns all OWL classes (URIs, labels, comments) in an ontology graph.

call_tool('get_graph_classes', {
    'ontology_id': ontology_id,
})


{
  "statusCode": 200,
  "body": "{\"ontologyId\": \"f972955d-db45-4bac-968a-80e0c3d16d4c\", \"classes\": [{\"uri\": \"http://example.com/ontology/f972955d-db45-4bac-968a-80e0c3d16d4c/AdminCode\", \"label\": \"Admin Code\", \"comment\": \"Administrative code record from the ODH system. Stores administrative system codes and identifiers used to classify and route insurance data across 22+ admin systems (e.g., ULA, ORD, DRIA, MCCV, CFO, CK4, IPAS). Uses DynamoDB-style pk/sk/data structure.\"}, {\"uri\": \"http://example.com/ontology/f972955d-db45-4bac-968a-80e0c3d16d4c/Coverage\", \"label\": \"Coverage\", \"comment\": \"Insurance coverage details within a policy. Represents the specific protections, benefits, and terms provided under an insurance contract, including coverage codes, descriptions, limits, deductibles, and premiums per ACORD CoverageType standards.\"}, {\"uri\": \"http://example.com/ontology/f972955d-db45-4bac-968a-80e0c3d16d4c/CoverageProduct\", \"label\": \"Coverage Produ

In [11]:
# ── get_graph_properties ─────────────────────────────────────────────────────
# Returns all RDF properties (URIs, labels, comments) in an ontology graph.

call_tool('get_graph_properties', {
    'ontology_id': ontology_id,
})


{
  "statusCode": 200,
  "body": "{\"ontologyId\": \"f972955d-db45-4bac-968a-80e0c3d16d4c\", \"properties\": [{\"uri\": \"http://example.com/ontology/f972955d-db45-4bac-968a-80e0c3d16d4c/AdminCode/data\", \"label\": \"data\", \"comment\": \"JSON data payload containing admin code attributes and values for the administrative system record.\", \"domain\": \"http://example.com/ontology/f972955d-db45-4bac-968a-80e0c3d16d4c/AdminCode\", \"range\": \"http://www.w3.org/2001/XMLSchema#string\", \"mapsToColumn\": \"admincode.data\", \"mapsToTable\": \"\"}, {\"uri\": \"http://example.com/ontology/f972955d-db45-4bac-968a-80e0c3d16d4c/AdminCode/pk\", \"label\": \"partition key\", \"comment\": \"Primary partition key identifying the admin code record. Typically contains the admin system identifier.\", \"domain\": \"http://example.com/ontology/f972955d-db45-4bac-968a-80e0c3d16d4c/AdminCode\", \"range\": \"http://www.w3.org/2001/XMLSchema#string\", \"mapsToColumn\": \"admincode.pk\", \"mapsToTable\":

In [12]:
# ── get_ontology_from_neptune ────────────────────────────────────────────────
# Retrieves the full ontology (classes, properties, mappings) for an ontology_id.
# The ontology_id is the UUID assigned when the ontology was created.

ontology_id = 'f972955d-db45-4bac-968a-80e0c3d16d4c'

call_tool('get_ontology_from_neptune', {
    'ontology_id': ontology_id,
})

{
  "statusCode": 200,
  "body": "{\"ontology_id\": \"f972955d-db45-4bac-968a-80e0c3d16d4c\", \"graph_uri\": \"http://example.com/ontology/f972955d-db45-4bac-968a-80e0c3d16d4c\", \"databases\": [], \"classes\": {\"http://example.com/ontology/f972955d-db45-4bac-968a-80e0c3d16d4c/AdminCode\": {}, \"http://example.com/ontology/f972955d-db45-4bac-968a-80e0c3d16d4c/Coverage\": {}, \"http://example.com/ontology/f972955d-db45-4bac-968a-80e0c3d16d4c/CoverageProduct\": {}, \"http://example.com/ontology/f972955d-db45-4bac-968a-80e0c3d16d4c/FinancialActivity\": {}, \"http://example.com/ontology/f972955d-db45-4bac-968a-80e0c3d16d4c/FinancialStatement\": {}, \"http://example.com/ontology/f972955d-db45-4bac-968a-80e0c3d16d4c/Holding\": {}, \"http://example.com/ontology/f972955d-db45-4bac-968a-80e0c3d16d4c/InvestProduct\": {}, \"http://example.com/ontology/f972955d-db45-4bac-968a-80e0c3d16d4c/Party\": {}, \"http://example.com/ontology/f972955d-db45-4bac-968a-80e0c3d16d4c/PolicyProduct\": {}, \"http:/

In [13]:
# # ── persist_to_neptune ───────────────────────────────────────────────────────
# # Writes RDF n-quad data to Neptune via SPARQL INSERT DATA.
# # Format: <subject> <predicate> <object> <named-graph> .

# nquad_data = """<http://example.com/test/subject1> <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://www.w3.org/2002/07/owl#Class> <http://example.com/test/graph> .
# <http://example.com/test/subject1> <http://www.w3.org/2000/01/rdf-schema#label> "TestClass" <http://example.com/test/graph> ."""

# call_tool('persist_to_neptune', {
#     'nquad_data': nquad_data,
# })
